# Линейная регрессия на реальных данных (привет, Малые Кармакулы)

В этом ноутбуке мы переходим от синтетических данных к реальным наблюдениям метеостанции **Малые Кармакулы** (архипелаг Новая Земля, [архив ВНИИГМИ-МЦД](http://meteo.ru/data/)) и решаем на них две классические задачи машинного обучения:

- **регрессия** — восстановить температуру воздуха `ta` по остальным метеорологическим величинам с помощью линейной регрессии;
- **классификация** — определить, безоблачно ли небо (`tcc == 0`), с помощью логистической регрессии.

Данные берём уже подготовленными: их чтение, очистка и первичный анализ разобраны на прошлом занятии в ноутбуке `Lect02_EDA_demo.ipynb`.

## Полезные ссылки
- <a href="https://numpy.org/doc/stable/">Документация NumPy</a>
- <a href="https://pandas.pydata.org/">Документация pandas</a>
- <a href="https://matplotlib.org/">Документация Matplotlib</a>
- <a href="https://scikit-learn.org/stable/">Документация scikit-learn</a>
- <a href="https://scikit-learn.org/stable/modules/linear_model.html">Линейные модели в scikit-learn</a>
- <a href="https://scikit-learn.org/stable/modules/model_evaluation.html">Метрики качества моделей в scikit-learn</a>

## Дополнительные материалы
- Ноутбук `Lect02_EDA_demo.ipynb` — разведочный анализ этих же данных
- Ноутбук `Lect03_LR_synthetic_data.ipynb` — линейная регрессия на синтетических данных
- [Метрики регрессии: MSE, RMSE, MAE, R²](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)
- [Матрица ошибок (confusion matrix)](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix)
- [Учебник «An Introduction to Statistical Learning»](https://www.statlearning.com/)

## Импорт библиотек

Подключаем всё необходимое: работу с массивами, таблицами и графиками, а также модели и метрики из scikit-learn.

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [scikit-learn](https://scikit-learn.org/stable/api/index.html) (модули [sklearn.linear_model](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.linear_model) и [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## Загрузка подготовленных данных

Считываем файл `EDA_demo_data/karmakuly_processed.csv` — результат обработки данных на прошлом занятии: в нём уже собрана единая метка времени, признаки переименованы и приведены к числовому типу. Колонку `Date` сразу делаем индексом временного ряда, чтобы далее было удобно брать выборки по датам.

**Инструменты:** [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) — параметр `parse_dates` разбирает колонку с датой, `index_col` назначает её индексом; [DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) — просмотр первых строк таблицы.

In [ ]:
df = pd.read_csv('EDA_demo_data/karmakuly_processed.csv', parse_dates=['Date'], index_col='Date')
df.head()

## Разделение признаков и целевой переменной. Обучение линейной регрессии

Задача ставится так: по результатам наблюдений нужно предсказать **температуру воздуха** `ta` — это целевая переменная (`target`), а все остальные измеряемые величины — признаки.

Из признаков сразу исключаем три столбца:

- `tg` — температура поверхности почвы: в этих данных она (почти) вся отсутствует;
- `td` — температура точки росы и `psl` — давление на уровне моря: они почти дублируют другие признаки, а точка росы вдобавок напрямую связана с температурой воздуха.

Оставшиеся строки с пропусками удаляем методом `dropna()`, чтобы у модели не было «дырок» в данных.

**Инструменты:** [DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html), [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html), [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) и её метод [fit](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.fit).

In [ ]:
target = 'ta'  # целевая переменная — температура воздуха

# tg, td и psl исключаем: они почти дублируют другие признаки
df_Xy = df.copy().drop(columns=['tg', 'td', 'psl']).dropna()

y = df_Xy[target]                 # целевая переменная
X = df_Xy.drop(target, axis=1)    # матрица признаков

model = LinearRegression()
model.fit(X, y)

### Матрица признаков

Проверяем, что именно подаётся на вход модели: `X` — это таблица «наблюдение × признак» (строки — моменты времени, столбцы — метеорологические величины), а `y` — вектор наблюдённых значений температуры воздуха.

**Инструменты:** отображение объекта [DataFrame](https://pandas.pydata.org/docs/reference/frame.html) средствами Jupyter.

In [ ]:
X  # матрица признаков: строки — моменты времени, столбцы — признаки

## Применение модели и оценка качества

Применяем обученную модель к тем же данным, на которых она обучалась. Так мы пока лишь смотрим, насколько хорошо регрессия описывает известные ей наблюдения; честная оценка качества потребует отложенной выборки (train/test split) и кросс-валидации, о которых речь пойдёт дальше.

На графике по осям отложены наблюдённая и предсказанная температура воздуха, каждая шестиугольная корзина (hexbin) собирает точки с близкими значениями, а её цвет — число наблюдений в корзине. Чёрная линия `y = x` соответствует идеальному прогнозу: точки выше неё — модель занижает температуру, ниже — завышает.

Качество оцениваем тремя способами: коэффициент корреляции $r$, коэффициент детерминации $R^2$ (для парной линейной связи совпадает с $r^2$) и корень из среднеквадратичной ошибки RMSE.

**Инструменты:** [LinearRegression.predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.predict), [matplotlib.pyplot.hexbin](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hexbin.html), [pyplot.gca](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.gca.html), [numpy.corrcoef](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html), [sklearn.metrics.r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html), [sklearn.metrics.mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html).

In [ ]:
y_pred = model.predict(X)

plt.figure()
#plt.scatter(y, y_pred)
plt.hexbin (y, y_pred, mincnt=1, gridsize=50)

plt.axis('equal')
ax = plt.gca()
lims = [
       np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
       np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
       ]
plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)  # линия y = x, идеальный прогноз
plt.xlim(lims)
plt.ylim(lims)
plt.grid()
plt.xlabel('Наблюдённая температура воздуха, °C')
plt.ylabel('Предсказанная температура воздуха, °C')
plt.title('Линейная регрессия: наблюдённая и предсказанная температура (hexbin)')
plt.colorbar(label='Число наблюдений')

r = np.corrcoef(y_pred, y)[0,1]
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)

print ('corr coeff is ', r)
print ('corr coeff^2 is ', r**2)

print ('R2 is ', r2)
print ('RMSE is ', np.sqrt (mse))

## Коэффициенты регрессии

Коэффициент при каждом признаке показывает, на сколько градусов изменится предсказанная температура воздуха `ta` при увеличении этого признака на единицу (при неизменных значениях остальных признаков). Знаки и относительные величины коэффициентов позволяют судить о физическом смысле модели и о том, какие признаки для неё важнее.

**Инструменты:** атрибут [LinearRegression.coef_](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.coef_), [DataFrame.plot.bar](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html).

In [ ]:
coefs = pd.DataFrame(model.coef_, X.columns, columns=['coef']) 
display(coefs)
ax = coefs.plot.bar()
ax.set_xlabel('Признак')
ax.set_ylabel('Коэффициент регрессии')
plt.title('Коэффициенты линейной регрессии')
plt.grid(axis='y', alpha=0.3)

## Логистическая регрессия: задача классификации

Теперь на тех же данных решим задачу **классификации**. Целевая переменная бинарная: «безоблачно» — если общее количество облачности равно нулю (`tcc == 0`), и «облачно» — во всех остальных случаях. В отличие от линейной регрессии, логистическая регрессия предсказывает не число, а вероятность принадлежности к классу.

Признаки формируем так же, как и раньше (исключаем `tg`, `td` и `psl`), только вместо `ta` целевую переменную берём из столбца `tcc`, который в признаки уже не попадает.

**Инструменты:** [sklearn.linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [sklearn.metrics.ConfusionMatrixDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay

df_Xy = df.copy().drop(columns=['tg', 'td', 'psl']).dropna()

y = df_Xy['tcc'] == 0  # True — безоблачно (tcc = 0), False — облачно

X = df_Xy.drop(columns=['tcc'])  # признаки: все остальные величины, включая ta

### Обучение классификатора

Обучаем логистическую регрессию на той же матрице признаков. Параметр `max_iter=1000` задаёт предельное число итераций численного оптимизатора (по умолчанию 100): на реальных данных его часто не хватает, и алгоритм не успевает сойтись.

**Инструменты:** [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) и её метод [fit](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.fit).

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

### Предсказание классов

Метод `predict` возвращает не вероятности, а уже принятое решение — метку класса (`True` — безоблачно, `False` — облачно). Сами вероятности классов при необходимости можно получить отдельным методом `predict_proba`.

**Инструменты:** [LogisticRegression.predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict), [LogisticRegression.predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba).

In [ ]:
y_predicted = model.predict(X)  # метки классов: True — безоблачно, False — облачно

### Матрица ошибок (confusion matrix)

Строим матрицу ошибок: по строкам отложены истинные классы, по столбцам — предсказанные моделью, а в ячейках стоит число наблюдений. Диагональ — правильные ответы, вне диагонали — ошибки двух типов: ложные срабатывания (предсказали «безоблачно», а на самом деле облачно) и пропуски. По этой матрице считаются точность, полнота и другие метрики классификации.

**Инструменты:** [ConfusionMatrixDisplay.from_estimator](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator), [sklearn.metrics.confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).

In [ ]:
disp = ConfusionMatrixDisplay.from_estimator(
    model, X, y,
    display_labels=['Облачно (tcc > 0)', 'Безоблачно (tcc = 0)'],
    cmap='Blues')

disp.ax_.set_xlabel('Предсказанный класс')
disp.ax_.set_ylabel('Истинный класс')
disp.ax_.set_title('Матрица ошибок логистической регрессии')

### Метрики качества классификации

Считаем наиболее употребительные метрики для предсказанного бинарного класса «безоблачно» (`True`). Обозначения берём по клеткам матрицы ошибок из предыдущей ячейки:

- $\text{TP}$ — истинные «безоблачно», предсказаны «безоблачно»;
- $\text{FP}$ — на самом деле «облачно», предсказаны «безоблачно» (ложное срабатывание);
- $\text{FN}$ — на самом деле «безоблачно», предсказаны «облачно» (пропуск);
- $\text{TN}$ — истинные «облачно», предсказаны «облачно».

Важно: класс «безоблачно» редкий (примерно 9 % наблюдений), поэтому **accuracy** здесь почти ни о чём не говорит — модель, которая всегда отвечает «облачно», получит около 0.91.

**Accuracy** — доля всех верных ответов:

$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

**Balanced accuracy** — полнота, усреднённая по двум классам (не «обманывается» дисбалансом классов):

$$\text{Balanced accuracy} = \frac{1}{2}\left(\frac{\text{TP}}{\text{TP} + \text{FN}} + \frac{\text{TN}}{\text{TN} + \text{FP}}\right)$$

**Precision** (точность) — какая доля предсказанных «безоблачно» действительно безоблачна:

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

**Recall** (полнота) — какую долю реальных «безоблачно» модель нашла:

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

**F1** — среднее гармоническое precision и recall (компромисс между ними):

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}}$$

**ROC-AUC** — площадь под ROC-кривой, по осям которой отложены полнота $\text{TPR} = \text{TP} / (\text{TP} + \text{FN})$ и доля ложных срабатываний $\text{FPR} = \text{FP} / (\text{FP} + \text{TN})$. Она же равна вероятности, что случайно взятому «безоблачно» модель дала большую вероятность класса, чем случайно взятому «облачно»:

$$\text{ROC-AUC} = \int_0^1 \text{TPR} d(\text{FPR}) = P(s_+ > s_-)$$

0.5 — случайное угадывание, 1.0 — идеальное разделение.

**PR-AUC** (average precision) — площадь под кривой «полнота – точность»:

$$\text{PR-AUC} = \sum_{n} \left(R_n - R_{n-1}\right) P_n$$

где $R_n$ и $P_n$ — полнота и точность на $n$-м пороге вероятности; при редком классе информативнее, чем ROC-AUC.

**Инструменты:** [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics) — [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [balanced_accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html), [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html), [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html), [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [average_precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html); сводку по классам одним вызовом даёт [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html).

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score)

print(f'Доля класса «безоблачно» в данных: {y.mean():.1%}')

y_proba = model.predict_proba(X)[:, 1]  # вероятность класса «безоблачно» (True)

metrics = {
    'Accuracy (доля верных ответов)': accuracy_score(y, y_predicted),
    'Balanced accuracy (полнота, усреднённая по классам)': balanced_accuracy_score(y, y_predicted),
    'Precision (доля верных среди предсказанных «безоблачно»)': precision_score(y, y_predicted),
    'Recall (доля найденных «безоблачно»)': recall_score(y, y_predicted),
    'F1 (баланс precision и recall)': f1_score(y, y_predicted)
}

display(pd.Series(metrics).round(3).to_frame('значение'))

### ROC-кривая

ROC-кривая показывает, как меняется качество решения при изменении порога отсечения вероятности (в `predict()` он по умолчанию равен 0.5). Для каждого порога считаются две величины:

- $\text{TPR}$ (полнота, sensitivity) — доля найденных «безоблачно»: $\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$;
- $\text{FPR}$ — доля ложных срабатываний: $\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$.

Кривая изображает зависимость $\text{TPR}$ от $\text{FPR}$ при движении порога: чем ближе кривая к левому верхнему углу, тем лучше модель, а диагональ отвечает модели, не несущей информации. Площадь под кривой (AUC, она же статистика Манна–Уитни) равна вероятности, что случайно взятому «безоблачно» модель приписала большую вероятность класса, чем случайно взятому «облачно»: $\text{AUC} = P(s_+ > s_-)$.

Для сравнения строим случайный baseline — `DummyClassifier(strategy='stratified')`: он «угадывает» классы случайно, с их повторяемостью в данных, и потому даёт AUC около 0.5, а его точки ложатся прямо на диагональ.

Полезно помнить две вещи. Во-первых, при доле редкого класса 9 % ROC-кривая выглядит оптимистично: отрицательных случаев 116 754, поэтому «размен» ошибок по FPR очень дешёвый, и рядом с ROC-AUC стоит смотреть PR-AUC (average precision) из предыдущей ячейки. Во-вторых, наклон ROC в точке порога равен отношению правдоподобия, так что кривая — это «портрет» того, какие решения реализует модель на всех порогах: точка порога 0.5 по умолчанию (отмечена звёздочкой) лежит в области малых $\text{FPR}$ и малых $\text{TPR}$ — та самая «осторожная» рабочая точка, о которой шла речь при обсуждении дисбаланса.

**Инструменты:** [sklearn.metrics.roc_curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html), [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [RocCurveDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html) (готовый график одной строкой), [sklearn.dummy.DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html), [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

# Случайный baseline: классы «угадываются» случайно, с их повторяемостью в данных
baseline = DummyClassifier(random_state=0).fit(X, y) #strategy='stratified', 
y_proba_baseline = baseline.predict_proba(X)[:, 1]

# Точка, которую выбирает порог по умолчанию 0.5 (из матрицы ошибок предыдущей ячейки)
tn_, fp_, fn_, tp_ = confusion_matrix(y, y_predicted).ravel()
fpr_default, tpr_default = fp_ / (fp_ + tn_), tp_ / (tp_ + fn_)

plt.figure()
for name, proba, fmt in (('Логистическая регрессия', y_proba, '-'),
                         ('Случайный baseline (DummyClassifier)', y_proba_baseline, 'o--')):
    fpr, tpr, _ = roc_curve(y, proba)  # y_proba посчитана в предыдущей ячейке
    plt.plot(fpr, tpr, fmt, ms=5, label=f'{name}: AUC = {roc_auc_score(y, proba):.3f}')

plt.plot([0, 1], [0, 1], 'k:', lw=1, label='Диагональ: информации нет (AUC = 0.5)')
plt.plot(fpr_default, tpr_default, 'k*', ms=13, label='Порог 0.5 (predict по умолчанию)')

plt.xlabel('Доля ложных срабатываний, FPR = FP / (FP + TN)')
plt.ylabel('Полнота, TPR = TP / (TP + FN)')
plt.title('ROC-кривая: логистическая регрессия и случайный baseline')
plt.legend(loc='lower right')
plt.grid()